In [ ]:
import shapely
from MobilityHubDataObjects import *
import pandas as pd
import geopandas as gpd
import folium
import numpy as np
from pyproj import Transformer
import datetime as dt

map_area = gpd.read_file("./rawData/LA_City_Boundary/City_Boundary.shp").to_crs("EPSG:4326").loc[0, "geometry"]
#map_area = gpd.read_file("./rawData/LA_Times_Neighborhood_Boundaries.geojson").loc[65, "geometry"]
#map_area = gpd.read_file("./rawData/santacruz_county.geojson").to_crs(4326).loc[0,"geometry"]
map_area

In [ ]:
citybikes_instance = CityBikesDataObject("http://api.citybik.es/")
citybikes_instance.load_data(map_area)

In [ ]:
citybikes_instance.data_object

In [ ]:
gtfs_instance = GTFSDataObject(
    "./gtfs/santacruz",
    "https://transit.land/api/v2/rest/feeds.json",
    dt.timedelta(days=10),
    dt.time(hour=10), #TODO: need to handle tz
    dt.time(hour=15),
    min_headway=9000,
    api_key_path="./rawData/TRANSITLAND_KEY",
)
gtfs_instance.load_data(map_area)

In [ ]:
gtfs_instance.df_feeds_metadata

In [6]:
fta_instance = FTAFacilityInventoryDataObject(
    "./rawData/2022 Facility Inventory.xlsx",
    column_filter={
        "Facility Type": [
            "Underground Fixed Guideway Station",
            "At-Grade Fixed Guideway Station",
            "Elevated Fixed Guideway Station",
        ]
    }
)
fta_instance.load_data(map_area)

In [ ]:
afdc_instance = AFDCApiDataObject(constants.AFDC_DATA_URL, "./cache/afdc_cache.geojson", "./rawData/AFDC_API_KEY")

afdc_instance.load_data(map_area)

In [8]:
osm_instance = OSMDataObject("./cache/osmnx_cache", {"amenity": ["bicycle_parking"]})
osm_instance.load_data(map_area)

In [ ]:
osm_instance.data_object

In [10]:
ejscreen_instance = EJScreenDataObject("./rawData/ejscreen_norcal.geojson")
ejscreen_instance.load_data(map_area)

In [11]:
ipcd_instance = IPCDDataObject("./rawData/NTAD_Intermodal_Passenger_Connectivity_Database_-3991686045944498549.geojson")
ipcd_instance.load_data(map_area)

In [ ]:
display_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
)
ejscreen_instance.get_folium_plot().add_to(display_map)
afdc_instance.get_folium_plot().add_to(display_map)
fta_instance.get_folium_plot().add_to(display_map)
osm_instance.get_folium_plot().add_to(display_map)
#ipcd_instance.get_folium_plot().add_to(display_map)
gtfs_instance.get_folium_plot().add_to(display_map)
citybikes_instance.get_folium_plot().add_to(display_map)
display_map